# Load your workbench data into a notebook

**Learning objectives**

- Read the datasets you uploaded or cleaned in Furina's workbench from Python.
- Compare a model you build in a notebook with a run you trained in the workbench, on the same data.

**Dependencies:** tier 2 — pandas, plus the small `furina.py` helper that sits next to this notebook. It only reads: it lists and downloads, and it never uploads, trains or deletes anything.

**Where this works.** In the **browser preview** (JupyterLite) the helper talks to the Furina site you opened the notebook from. Your **personal notebook server** is deliberately cut off from the rest of the site, so there the helper explains that it cannot connect. To use your own data on that server, download the CSV from the workbench and upload it into JupyterLab with the upload button.

In [ ]:
import pandas as pd

import furina

print("Furina API:", furina.api_url())

## 1. List the datasets in the workbench

Each entry is one dataset: an upload, or a cleaned copy saved from Data Preprocessing. If the list is empty, upload a CSV on the workbench's Data Preprocessing page first.

In [ ]:
try:
    datasets = furina.list_datasets()
except furina.FurinaError as error:
    datasets = []
    print(error)

for entry in datasets:
    print(f"{entry['name']!r}: {entry['n_rows']} rows x {entry['n_cols']} columns (id {entry['id']})")
if not datasets:
    print("No datasets to show.")

## 2. Load one into pandas

`load_dataset` accepts a dataset's id or its exact name. The cell below takes the newest dataset; replace `chosen` with a name from the list above to pick another.

In [ ]:
frame = None
if datasets:
    chosen = datasets[0]["id"]
    frame = furina.load_dataset(chosen)
    print(frame.shape)
    print(frame.head())
else:
    print("Nothing loaded: there is no dataset to read from here.")

**Expected behaviour.** The table has the same rows and columns that the workbench's column profile reports for that dataset, because both read the same stored CSV.

## 3. Compare with a workbench run

Runs carry their test metrics, so a model you build in a notebook can be judged against one trained in the workbench. The comparison is only fair on the same rows: the workbench shuffles and splits with the seed shown in the run, so use the same test size, seed and stratification if you rebuild its split.

In [ ]:
runs = []
if frame is not None:
    try:
        runs = furina.list_runs(chosen)
    except furina.FurinaError as error:
        print(error)
for run in runs[:5]:
    print(run["algorithm_label"], run["primary_metric_name"], run["primary_metric"])
if frame is not None and not runs:
    print("No runs yet for this dataset: train one on the Data Training page.")

## Exercises

1. Pick a dataset and count the missing values in each column with `frame.isna().sum()`. Do the numbers match the workbench's column profile?
2. Find the run with the best primary metric for this dataset, then write down its seed and test size. Rebuild its train/test split in pandas.

## Common errors

- **"could not be reached"**: you are on your personal notebook server, which cannot reach the site by design, or the workbench is not running. Use the browser preview, or upload the CSV yourself.
- **"No dataset has the id or name …"**: names must match exactly. Copy the id from the list instead.

## Summary

The workbench and your notebooks can share the same data. Notebooks give you full control over the code, and the workbench gives you a fast way to check it against a trained baseline.